# Begin

In [1]:
import os # @launchit.collect
import sys # @launchit.collect
import socket
import copy
from collections import namedtuple, defaultdict, Counter, deque # @launchit.collect
import random
import math
import datetime
import json # @launchit.collect
import pprint # @launchit.collect
import re 
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import pickle # @launchit.collect
import IPython 
from enum import StrEnum, auto # @launchit.collect
import multiprocessing as mp
import queue

import lark # @launchit.collect

from tqdm.notebook import tqdm

import numpy as np # @launchit.collect
import cupy as cp
import einops
import pandas as pd
import matplotlib.pyplot as plt
import scipy.special

import torch
import torch.nn as nn 
import torch.nn.functional as F
import torchvision.transforms.functional as VF
import torch.optim
import torch.multiprocessing as torch_mp
from torch.distributions import Categorical

import gymnasium as gym
import ale_py
from moviepy.video.io.ImageSequenceClip import ImageSequenceClip
import av

import optuna # @launchit.collect
from optuna.storages import JournalStorage # @launchit.collect
from optuna.storages.journal import JournalFileBackend # @launchit.collect
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
build_project_root_path = '${BUILD_PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect
sys.path.append(os.path.join(build_project_root_path, 'lib')) # @launchit.collect
import lang_utils as lu # @launchit.collect
import array_utils as au # @launchit.collect
from math_utils import RecursiveAverageFilter, RecursiveMovingAverageFilter
from logging_utils import *
from artifact_registry import * # @launchit.collect
import torch_utils
import launchit 
from hp_utils import * # @launchit.collect
from metrics_collector import RmqSummaryWriter, S3SummaryWriter
from autoincrement import Autoincrement
import command_listener
import ob_preprocessor

# Init

In [2]:
# @launchit.collect
class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    DOCKER_LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()

In [3]:
def create_config():
    config = namedtuple('Config', 
                        'host_name, ' +
                        'project_root_path, project_root_uri, model_group_uri, subproject_path, data_path, private_data_path, run_path, initrd_path, ' + 
                        'self_fname, self_name, metrics_suite_fname, ' +
                        'subproject_name,' +
                        'is_cuda, cuda_device, docker_registry, exec_mode, is_interactive')(
        host_name=socket.gethostname(),
        project_root_path=project_root_path,
        project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
        model_group_uri=None,
        subproject_path=os.path.abspath('.'),
        data_path=os.path.join(project_root_path, 'data'),
        private_data_path=None,
        run_path=None,
        initrd_path=None,
        self_fname=None,
        self_name=None,
        metrics_suite_fname=None,
        subproject_name=None,
        is_cuda=torch.cuda.is_available(),
        cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
        docker_registry='cr.selcloud.ru/neurolab',
        exec_mode=ExecMode.MASTER_NOTEBOOK,
        is_interactive=True,
    )
    
    if IPython.get_ipython() is None:
        module_fname = __file__
        module_basename = os.path.basename(module_fname)
        module_name, _ = os.path.splitext(module_basename)
        
        config = config._replace(self_fname=module_fname, self_name=module_name)
        config = config._replace(exec_mode=ExecMode.LAUNCH_MODULE)
    else:
        with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
            notebook_fname = json.load(cf).get('jupyter_session')

            if notebook_fname is None:
                notebook_fname = os.path.join(config.subproject_path, os.path.basename('${LAUNCHIT_FNAME}'))
                assert os.path.exists(notebook_fname)
            
            notebook_basename = os.path.basename(notebook_fname)
            notebook_name, notebook_ext = os.path.splitext(notebook_basename)
        
            m = re.match(r'(\w+)-Copy\d+$', notebook_name)
        
            if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook
    
            config = config._replace(self_fname=notebook_fname, self_name=notebook_name)
            
            is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None

            if is_launch:
                if os.path.exists(os.path.join(project_root_path, '.docker_launch')):
                    config = config._replace(exec_mode=ExecMode.DOCKER_LAUNCH_NOTEBOOK)
                else:
                    config = config._replace(exec_mode=ExecMode.LAUNCH_NOTEBOOK)
            else:
                assert config.exec_mode == ExecMode.MASTER_NOTEBOOK
    
    config = config._replace(is_interactive=config.exec_mode in [ExecMode.MASTER_NOTEBOOK, ExecMode.LAUNCH_NOTEBOOK])
    config = config._replace(subproject_name=os.path.basename(os.path.dirname(config.self_fname)))
    config = config._replace(model_group_uri=f'{config.project_root_uri}.{config.subproject_name}')
    config = config._replace(run_path=os.path.join(project_root_path, 'run', config.subproject_name))
    config = config._replace(initrd_path=os.path.join(project_root_path, 'run', config.subproject_name, 'initrd-' + config.self_name))
    config = config._replace(private_data_path=os.path.join(config.data_path, config.subproject_name))
    config = config._replace(metrics_suite_fname=os.path.join(config.run_path, config.self_name + '.metrics_suite.json'))
    return config

In [4]:
# @launchit.disable_worker
au.init()
LOG = Logging.get()
RNG = np.random.default_rng()
CONFIG = create_config()
LOG.app_name = CONFIG.self_name
LOG.enable('syslog', CONFIG.exec_mode == ExecMode.LAUNCH_MODULE)
LOG.enable('stdout', CONFIG.exec_mode in [ExecMode.MASTER_NOTEBOOK, ExecMode.LAUNCH_NOTEBOOK])
LOG.enable('verbose_stdout', CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK)
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)
os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)
os.makedirs(CONFIG.initrd_path, exist_ok=True)

CONFIG=
{'host_name': 'thinkbook',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'data_path': '/home/misha/dev/mine/neurolab/data',
 'private_data_path': '/home/misha/dev/mine/neurolab/data/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'initrd_path': '/home/misha/dev/mine/neurolab/run/18_rl/initrd-18n_explore_02',
 'self_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_explore_02.ipynb',
 'self_name': '18n_explore_02',
 'metrics_suite_fname': '/home/misha/dev/mine/neurolab/run/18_rl/18n_explore_02.metrics_suite.json',
 'subproject_name': '18_rl',
 'is_cuda': False,
 'cuda_device': 'cpu',
 'docker_registry': 'cr.selcloud.ru/neurolab',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



In [5]:
LaunchComponent = namedtuple('LaunchComponent', 'name version uri main_asset_fname')
    
@dataclass(slots=True)
class Hyperparameters:
    @dataclass(slots=True)
    class Env:
        is_episodic_life: bool = True

    env: Env = dataclasses.field(default_factory=Env)
    
    def launch_component(self):
        name = CONFIG.self_name
        return LaunchComponent(name=name, version=0,  uri=f'{CONFIG.model_group_uri}.{name}', main_asset_fname=CONFIG.self_fname)

HP = Hyperparameters()

In [6]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

# Agent

In [7]:
LoadAgentResult = namedtuple('LoadAgentResult', 'vision_head, encoder, agent, module, hp')

def load_agent(agent_id, artifact_registry, notebook_fname=None):
    coords = hp_parse_artifact_source(agent_id)

    meta = artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_classifier='meta', asset_ext='json', maven_group_id=coords.group_id)
    meta = json.loads(meta.decode('utf-8'))
    hp = meta['hypers']

    # Vision head
    vision_head_id = hp['vision_head']['parent']['model']
    vision_head_coords = hp_parse_artifact_source(vision_head_id)
    vision_head_notebook_data = artifact_registry.get_asset_content(
        vision_head_coords.model_name, 
        vision_head_coords.model_version, 
        asset_ext='ipynb', 
        maven_group_id=vision_head_coords.group_id,
    )
    vision_head_source_code = launchit.extract_source_code(io.BytesIO(vision_head_notebook_data), collect_inds=[None, 'export'])
    extra_imports = [
        'import ob_preprocessor'
    ]
    vision_head_source_code = '\n'.join(extra_imports) + '\n' + vision_head_source_code 
    vision_head_module = lu.make_module(f'{vision_head_id}_notebook', vision_head_source_code)
    vision_head_module.CONFIG = CONFIG
    vision_head_module.RNG = RNG
    vision_head_module.LOG = LOG
    vision_head_params = artifact_registry.get_asset_content(
        vision_head_coords.model_name, 
        vision_head_coords.model_version, 
        asset_classifier='vision_head_params', 
        asset_ext='json', 
        maven_group_id=coords.group_id,
    )
    vision_head_params = vision_head_module.VisionHead.Params(**json.loads(vision_head_params.decode('utf-8')))
    vision_head = vision_head_module.VisionHead(vision_head_params)

    # Encoder
    encoder_id = hp['encoder']['parent']['model']
    encoder_coords = hp_parse_artifact_source(encoder_id)
    encoder_notebook_data = artifact_registry.get_asset_content(
        encoder_coords.model_name, 
        encoder_coords.model_version, 
        asset_ext='ipynb', 
        maven_group_id=encoder_coords.group_id,
    )
    encoder_source_code = launchit.extract_source_code(io.BytesIO(encoder_notebook_data), collect_inds=[None, 'export'])
    encoder_module = lu.make_module(f'{encoder_id}_notebook', encoder_source_code)
    encoder_module.CONFIG = CONFIG
    encoder_module.RNG = RNG
    encoder_module.LOG = LOG
    encoder_params = artifact_registry.get_asset_content(
        encoder_coords.model_name, 
        encoder_coords.model_version, 
        asset_classifier='encoder_params', 
        asset_ext='json', 
        maven_group_id=coords.group_id,
    )
    encoder_params = encoder_module.Encoder.Params(**json.loads(encoder_params.decode('utf-8')))
    encoder = encoder_module.Encoder(encoder_params)

    # Agent
    if notebook_fname is None:
        notebook_data = artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='ipynb', maven_group_id=coords.group_id)
        source_code = launchit.extract_source_code(io.BytesIO(notebook_data), collect_inds=[None, 'export'])
    else:
        source_code = launchit.extract_source_code(notebook_fname, collect_inds=[None, 'export'])
        
    # print(source_code)
    # workaround for missing imports
    extra_imports = [
        'from torch.distributions import Categorical',
        'import math',
        'from torch.nn.attention import SDPBackend, sdpa_kernel',
        'import torch_utils',
        'import ob_preprocessor',
    ]
    source_code = '\n'.join(extra_imports) + '\n' + source_code 
    module = lu.make_module(f'{agent_id}_notebook', source_code)
    module.CONFIG = CONFIG
    module.RNG = RNG
    module.LOG = LOG
    module.HP = HP
    agent_params = artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_classifier='agent_params', asset_ext='json', maven_group_id=coords.group_id)
    agent_params = module.Agent.Params(**json.loads(agent_params.decode('utf-8')))
    agent = module.Agent(agent_params)

    for model_name, model in zip(('vision_head', 'encoder', 'agent'), (vision_head, encoder, agent)):
        pt_data = artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_classifier=model_name, asset_ext='pt', maven_group_id=coords.group_id)
        
        with io.BytesIO(pt_data) as b:
            kwargs = {}
        
            if not CONFIG.is_cuda:
                kwargs['map_location'] = torch.device('cpu')

            state_dict = torch.load(b, **kwargs)
            model.load_state_dict(state_dict)
            model = model.to(CONFIG.cuda_device)
            model = torch.compile(model, fullgraph=True)
    
    return LoadAgentResult(
        vision_head=vision_head,
        encoder=encoder,
        agent=agent, 
        module=module, 
        hp=hp,
    )

In [8]:
# AGENT_ID = '18n_ppo_tr_frostbite_06:117'
# AGENT_ID = '18n_ppo_tr_frostbite_06:172'
# AGENT_ID = '18n_ppo_tr_frostbite_06:124'
# AGENT_ID = '18n_ppo_tr_frostbite_06:162'
# AGENT_ID = '18n_ppo_tr_frostbite_06:232'
# AGENT_ID = '18n_ppo_tr_frostbite_06:277'
# AGENT_ID = '18n_ppo_tr_frostbite_06:304'
# AGENT_ID = '18n_ppo_tr_frostbite_06:326'
# AGENT_ID = '18n_ppo_tr_frostbite_06:335'
# AGENT_ID = '18n_ppo_tr_frostbite_06:524'
AGENT_ID = '18n_ppo_tr_frostbite_06:529'
AGENT_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_06.ipynb'
# AGENT_NOTEBOOK_FNAME = None
lar = load_agent(AGENT_ID, ARTIFACT_REGISTRY, AGENT_NOTEBOOK_FNAME)
VISION_HEAD, ENCODER, AGENT, AGENT_MODULE = lar.vision_head, lar.encoder, lar.agent, lar.module

# Video capture

In [13]:
rams = {}
ram_id = None
# ram_id = 'com.develorium.neurolab.frostbite_ram:level5_101:1'
# ram_id = 'com.develorium.neurolab.frostbite_ram:level1_101:1'
# ram_id = 'com.develorium.neurolab.frostbite_ram:level1:1'
# ram_id = 'com.develorium.neurolab.frostbite_ram:level1:1:cls=exam1'
# ram_id = 'com.develorium.neurolab.frostbite_ram:level1:1:cls=exam2'
ram_id = 'com.develorium.neurolab.frostbite_ram:level5:1:cls=train5'

if ram_id is not None:
    coords = hp_parse_artifact_source(ram_id)
    ram_asset = ARTIFACT_REGISTRY.get_asset_content(coords.model_name, coords.model_version, asset_ext='pkl', asset_classifier=coords.asset_classifier, maven_group_id=coords.group_id)
    
    with io.BytesIO(ram_asset) as b:
        ram = pickle.load(b)

    rams[ram_id] = ram
else:
    ram = None
    rams = {}

ram_patches = None
# ram_patches = [
#     ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'],
#     ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'],
#     ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'],                                        
#     ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'],
#     ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'],
# ]
# ram_patches = [
#     ['no_score', 'three_lives', 'no_igloo'],
# ]
# ram_patches = [
#     ['no_score', 'nine_lives', 'no_igloo'],
# ]
# ram_patches = [
#     ['no_score', 'last_life', 'no_igloo'],
# ]
# ram_patches = [
#     ['eight_lives'],
# ]
# ram_patches = [
#     ['bailey_safe_random_spawn', 'last_life'],
# ]
# ram_patches = [
#     ['last_life'],
# ]
# ram_patches = [
#     ['no_score', 'last_life', 'no_igloo', 'bailey_safe_random_spawn']
# ]
# ram_patches = [
#     ['no_score', 'three_lives', 'no_igloo', 'temperature_45'],
# ]

VISION_HEAD.eval()
ENCODER.eval()
AGENT.eval()
video_file_name = AGENT_MODULE.generate_video_file_name('/home/misha/tmp')

with torch.no_grad():
    capture_result = AGENT_MODULE.capture_video_of_test_rollout(
        VISION_HEAD,
        ENCODER,
        AGENT, 
        max_steps_count=10_000, 
        # random_seed=61,
        # random_seed=86,
        # random_seed=42 + 32 + 1,
        video_file_name=video_file_name,
        rams=rams,
        ram_patches=ram_patches,
        capture_preprocessed_obs=False,
        break_on_level_passed=False,
        # is_lossless=True,
        capture_values=True,
        # capture_action_plan_logits=True,
    )

capture_result[0], capture_result[1]['game']

('/home/misha/tmp/video-18n_explore_02-launch0-2026.09.23-12:26:30.mp4',
 defaultdict(int,
             {'random_seed': None,
              'reward': 200.0,
              'frames_count': 670,
              'steps_count': 162,
              'levels_passed': 0}))

In [10]:
x = np.array([3, 4, 4, 4, 4, 4, 2, 3, 4, 4, 3, 4, 4, 4, 5])
len(x), x.mean()

(15, np.float64(3.7333333333333334))